<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 1 &middot; Set up and validate</div><div style="font-size:15px;color:#cfd4ff;">Set the stage &middot; about 4 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 1. Set up and validate

**Time:** about 4 minutes. This is the first of three "set the stage" notebooks, then we start the climb.

**The question we're starting with:** how do I know my setup *really* works before I build on top of it?

Before we climb anything, let's make sure our camp is set up. On the self-guided path, `infra/deploy.sh` built the environment. In Skillable, the lifecycle already did that work. Both should leave the same Foundry project, model deployments, tracing and v0 agent. Now we check them using code.

Why bother? Because a skipped check turns into a confusing error three notebooks from now. A few minutes here saves a lot of head-scratching later.

By the end we'll have a short scoreboard of PASS and FAIL lines, and we'll know exactly what's ready.

> **How to use these notebooks:** run one cell at a time with **Shift+Enter**, and read the text before you run the code. Open the **Outline** view to see the numbered sections. They tell the story of the whole lab.


---

## 1.1 What are we checking, and how do I keep score?

All the notebooks share a small helper file, `src/lab.py`. It handles plumbing like reading `.env` and connecting. The interesting Foundry calls stay right here in the notebooks, so we can see what's happening.

The next cell also sets up a tiny `record()` function. Every check we run reports back to it, and we'll print one scoreboard at the end.

In [ ]:
import os
import sys
from pathlib import Path

# Our shared helpers live in src/. Add it to Python's search path so we can import them.
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))

import lab

results = []  # one (name, passed, detail) row per check


def record(name, passed, detail=""):
    """Remember a check's outcome and print it right away."""
    results[:] = [row for row in results if row[0] != name]  # rerunning a check replaces its old result
    results.append((name, passed, detail))
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {name}" + (f"  ({detail})" if detail and not passed else ""))

from importlib.metadata import version
from packaging.version import Version
sdk_ready = Version(version("azure-ai-projects")) >= Version("2.8.0")
record("Foundry SDK supports this lab", sdk_ready, "azure-ai-projects 2.8.0 or later is required")
if not sdk_ready:
    raise RuntimeError("Prepare the documented Python environment first. In Skillable, ask the facilitator to repair the VM image.")

## 1.2 Is my `.env` filled in?

The deployment wrote a `.env` file at the top of the repo. It holds our project endpoint, agent name and a few other settings. It stays on our machine and is never committed.

We sign in with our own Azure identity, so there are **no API keys** in it. That's safer, and it's the pattern we'd use in production.

In [ ]:
# Which settings did the deploy script leave us? Anything missing is listed here.
missing = lab.missing_env()
record("our .env has every setting we need", not missing, f"missing: {missing}")

**Expected:** `PASS`. Self-guided: recover the settings from the completed deployment before retrying; `deploy.sh` creates a **new resource group on every run**, including with `--skip-agent`. Skillable: ask the facilitator to rerun the lifecycle with the same lab instance ID. Do not run `deploy.sh` in Skillable.

## 1.3 Can I reach my project?

Now let's connect. `lab.connect()` gives us two clients: one for the Foundry project (agents, datasets, evaluators) and one OpenAI-compatible client for calling models and running evals.

In [ ]:
try:
    project_client, openai_client = lab.connect()
    next(iter(project_client.deployments.list()), None)  # a real request, not just client construction
    record("we can connect to our Foundry project", True)
    print("Project endpoint:", os.environ["FOUNDRY_PROJECT_ENDPOINT"])
except Exception as error:
    record("we can connect to our Foundry project", False, str(error)[:120])
    raise

## 1.4 Which models did I deploy?

We asked for six deployments: two chat models and four **model router** editions. A router is one endpoint that picks a model for each request. We'll meet them properly in Module 6, but let's confirm they exist.

**Think:** the four `mr-` deployments are the same router set up four different ways. If you had to guess, which one will be cheapest, and which one will be the strongest?

In [ ]:
expected = [lab.FRONTIER_MODEL, lab.SMALL_MODEL, *lab.ROUTERS]

# Ask the project which deployments it knows about.
deployed = sorted(d.name for d in project_client.deployments.list())
print("Deployed:", deployed)

not_found = [name for name in expected if name not in deployed]
record("all six model deployments exist", not not_found, f"not found: {not_found}")
snapshot = lab.record_deployment_configuration()
print("Recorded actual model versions, routing settings and capacity at", snapshot["recorded_at"])

## 1.5 Am I allowed to call them?

Deployments existing is one thing, being *allowed* to call them is another. Role assignments can take a few minutes to arrive, so this check matters. We'll send a tiny request to a normal model and to a router.

Look at the router's reply carefully. It tells us **which model it chose**, which is a clue about how it works.

In [ ]:
for deployment in [lab.FRONTIER_MODEL, lab.SMALL_MODEL, *lab.ROUTERS]:
    try:
        reply = openai_client.responses.create(model=deployment, input="Say hello in five words.")
        # For a router, response.model names the model it picked behind the scenes.
        print(f"{deployment} -> {reply.output_text!r}  (answered by: {reply.model})")
        record(f"we can call {deployment}", True)
    except Exception as error:
        record(f"we can call {deployment}", False, str(error)[:120])

**Think:** why might it be useful for a router to tell us which model it picked? Hold that thought. It comes back when we measure cost.

## 1.6 Is my v0 agent there?

`infra/deploy.sh` created **TrailMate v0**: a frontier model, a file search tool over the product manuals, and very plain instructions. Let's look at what it was given.

In [ ]:
# Read the v0 ID saved by setup. Never assume the latest version is our baseline.
versions = list(project_client.agents.list_versions(lab.AGENT_NAME))
record("the TrailMate agent exists", len(versions) >= 1, "run: python3 src/agent/01_build_agent.py")
if not versions:
    raise RuntimeError("No agent version exists. Finish the documented setup before continuing.")

if versions:
    state = lab.load_state()
    if "v0_version" not in state:
        raise RuntimeError("Setup has not pinned v0. Run the documented agent bootstrap before continuing.")
    latest = next((v for v in versions if str(v.version) == state["v0_version"]), None)
    if latest is None:
        raise RuntimeError("The pinned v0 is missing. Ask the instructor or review self-guided setup; do not substitute latest.")
    print(f"Versions found: {len(versions)}  |  pinned v0 is Foundry version {latest.version}")
    print("Model        :", latest.definition.model)
    print("Instructions :", repr(latest.definition.instructions))

Those instructions are one short sentence. That's on purpose. v0 is our starting camp, and we want a baseline that clearly has room to improve.

> **A note on version names.** Foundry assigns version IDs; our **v0, v1, v2 and v3** labels describe experiments. Reruns and candidates can consume IDs, so we save the actual mappings in `.lab-state.json` instead of assuming they are consecutive.

## 1.7 Can the agent search the manuals?

The agent answers from the product manuals using file search. That needs a **vector store**, which is a searchable index of the manuals.

In [ ]:
store = openai_client.vector_stores.retrieve(state["vector_store_id"])
record("all ten manuals are indexed", store.file_counts.completed == 10 and store.file_counts.failed == 0,
       str(store.file_counts))
attached = any(store.id in (getattr(tool, "vector_store_ids", None) or []) for tool in latest.definition.tools)
record("v0 searches this vector store", attached)

## 1.8 Is tracing connected?

Every time our agent runs, we want a **trace**: a record of what it did, step by step. Traces are the raw material for Module 4, where Foundry reads them and tells us what's going wrong. They land in Application Insights, so let's check the project is connected to one.

In [ ]:
try:
    connection_string = project_client.telemetry.get_application_insights_connection_string()
    record("tracing is connected to Application Insights", bool(connection_string))
except Exception as error:
    record("tracing is connected to Application Insights", False, str(error)[:120])

## 1.9 Can I trust my test data?

A score is only as trustworthy as the questions behind it. We have three sets of test cases (practice, dev and test), and every expected answer quotes the manuals word for word. `src/02_validate_data.py` checks that all the quotes are real and that the sets don't overlap.

This is a habit worth stealing: **test the tests** before trusting them.

In [ ]:
import importlib

# The file name starts with a number, so we load it by name instead of a normal import.
validate_data = importlib.import_module("02_validate_data")

# validate() returns (check name, passed, detail) for every data check.
for name, passed, detail in validate_data.validate():
    record(f"data: {name}", passed, detail)

## 1.10 So, is my camp ready?

Here's everything in one place.

In [ ]:
failed = [name for name, passed, _ in results if not passed]
print(f"{len(results) - len(failed)} of {len(results)} checks passed")

if failed:
    print("\nStill to fix:")
    for name in failed:
        print("  -", name)
    print("\nFix these before moving on. Each one has a hint in its own section above.")
else:
    print("\nOverall: PASS. Our camp is ready.")

**Expected:** all checks pass. If a few are red, that's normal on a first run (access can take a few minutes to arrive). Re-run the failed cell and move on once it's green.

## 1.11 What does a failure look like?

Confidence comes from seeing what failure looks like. Let's call a deployment that doesn't exist and read the error carefully. When something breaks later, we'll recognize this kind of message.

In [ ]:
try:
    openai_client.responses.create(model="gpt-9000-does-not-exist", input="hello")
except Exception as error:
    # Print only the first line so it's easy to read.
    print("Error we expected:", str(error).splitlines()[0][:200])

## 1.12 Why check one piece at a time? (the aha)

> **Aha:** "It doesn't work" is useless. "Check 5 failed" is a fix.
>
> Nothing here was hard. The value was that every seam (settings, access, models, agent, tracing, data) got its own name and its own PASS or FAIL. Most setup pain lives at those seams, not in the code, and naming each one turns a vague mystery into a one-line fix.

We also took away:

- Setup is a **checklist, not a hope**. Each check names one thing that can go wrong.
- Deployed doesn't mean reachable. Access and traces are separate pieces.
- We tested our **test data** too, because every score from here on depends on it.
- Failures have recognizable shapes. The one we just caused will look familiar later.

**Go further (optional):** add a check of your own to the scoreboard, for example "does the vector store hold 10 files?" Use `record()` the same way the other checks do.

**Next:** `02-model-selection.ipynb`. We have two models and four routers, so how do we choose?